In [1]:
from bs4 import BeautifulSoup
from src.loading import *
from src.preprocessing import *
import os
from flask import Flask, request, jsonify, render_template, send_from_directory
from src.model import *
import chromadb                              
from chromadb.errors import NotFoundError

c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Hp\OneDrive\Study\AI\SBS\Tasks\RAG\src\model.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2303.45it/s]


Download book if not exist 

In [3]:
url = 'https://www.alarabimag.com/books/22375-%D8%A3%D8%B1%D8%B6-%D8%B2%D9%8A%D9%83%D9%88%D9%84%D8%A7.html'
pdf_url , book_title = load(url)
txt_path = 'Data as text/أرض_زيكولا.txt'

if not os.path.exists(txt_path):
    # ------ get path for book 
    pdf_path = download_book(pdf_url , book_title)
else:
    # to run fast and do not download each time
    with open('Data as text/أرض_زيكولا.txt', 'r', encoding='utf-8') as file:
        data = file.read()


Fetching page .....
Featched successfully
PDF URL (fixed): https://www.alarabimag.com/download/22375-pdf
Book title: أرض زيكولا





مؤلف العمل:

عمرو عبد الحميد



حجم الكتاب:
5.1  ميغا بايت


عدد الصفحات:
434 صفحة


اللغة:
العربية


نوع العمل:
روايات عربية 


نوع الملف:
PDF


عدد القراءات:
406045 مرة



التقييم

(616) قارئ

4.12 من 5













قراءة الكتاب
تحميل الكتاب PDF


 
وصف رواية أرض زيكولا
رواية "أرض زيكولا" تُعد واحدة من أشهر أعمال الكاتب عمرو عبد الحميد. قدم فيها عبد الحميد مجموعة من الأجزاء، وهذا ما جعلها تحقق شهرة واسعة.تأسر هذه الرواية قلوب القراء وعشاق الخيال بفضل الذكاء والإبداع الذي ظهرا في كتابتها. تأخذنا الرواية في رحلة سردية مذهلة في عالم الخيال، مع تشابك بسيط بالواقع الصعب الذي نعيشه. إن هذا التوازن بين الواقع والخيال هو ما يميز هذا العمل الأدبي الرائع.تقدم القصة شخصية رئيسية تُدعى "خالد"، وهو شاب يعيش في زيكولا. في هذا العالم، يُجبر السكان على المشاركة في مباريات خطيرة تسمى "المباريات السنوية"، والتي يتعين على الخاسر فيها أن يُقتل. هذا النظام يعتمد على فكرة أن

Read text file after scanned the pdf and extract text usign OCR 

In [2]:
from src.preprocessing import *
from src.model import *
import os

txt_path = 'Data as text/أرض_زيكولا.txt'

# ---- Only run OCR if the text file doesn't exist ----
if not os.path.exists(txt_path):
    text = Extract_text('Books/أرض_زيكولا.pdf')
    data = clean_and_save(text, 'أرض_زيكولا') 
else:
    with open(txt_path, 'r', encoding='utf-8') as file:
        data = file.read()

In [ ]:
from flask import Flask, request, jsonify, render_template, send_from_directory
txt_path = 'Data as text/أرض_زيكولا.txt'
try:
    text                                            
except NameError:
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()


load_dotenv()
persist_dir = "./chroma_db"

client = chromadb.PersistentClient(path=persist_dir)
collection_name = "ard_zecola"


try:
    collection = client.get_collection(name=collection_name)
    # if collection empty delete it then create new 
    if collection.count() == 0:
        raise ValueError("Collection exists but is empty.")
except (NotFoundError, ValueError):
    chunks = chunking(text, chunk_size=500, overlap=80)
    embeddings = generate_embeddings(chunks)
    collection = store_in_vectorDB(chunks, embeddings, collection_name=collection_name, persist_dir=persist_dir)


# --------------------------------------------------------------------


app = Flask(__name__, template_folder='front end')

# default page 
@app.route('/')
def index():
    return render_template('index.html')

# ask question 
@app.route('/ask', methods=['POST'])
def ask():
    data = request.get_json()
    question = data.get('question', '').strip()
    if not question:
        return jsonify({'error': 'يرجى كتابة سؤال'}), 400

    # model
    result = query(question, collection, top_k=4)
    return jsonify({
        'answer': result['answer'],
        'sources': result['sources'],
        'source_ids': result['source_ids'],
        'question': question
    })



@app.route('/<path:filename>')
def serve_static_files(filename):
    return send_from_directory('front end', filename)


if __name__ == '__main__':
    # load model once 
    app.run(debug=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Jul/2026 02:58:23] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Jul/2026 02:58:23] "GET /style.css HTTP/1.1" 304 -
127.0.0.1 - - [12/Jul/2026 02:58:23] "GET /script.js HTTP/1.1" 304 -
127.0.0.1 - - [12/Jul/2026 02:58:23] "GET /images/background.jpg HTTP/1.1" 304 -
[2026-07-12 02:58:38,687] ERROR in app: Exception on /ask [POST]
Traceback (most recent call last):
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\api_core\grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\grpc\_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ^^^^^^^^^^^^^^^^
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\grpc\_interceptor.py", line 331, in _with_call
